# Pipeline
## Objective

The goal of this notebook is to build a complete, production-ready sklearn Pipeline that:
- Handles all data preprocessing steps (cleaning, feature engineering, encoding, and scaling)
- Processes both categorical and numerical features correctly
- Trains the final CatBoost model on the entire dataset
- Eliminates manual steps and ensures reproducibility
- Can be easily saved and deployed for inference on new raw data

### 1. Setup & Imports


In [3]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score

from catboost import CatBoostClassifier

import pickle
import warnings

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

In [4]:
def get_metrics(y_true, y_pred, y_proba):
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba)
    }


## 2. Load Raw Data

In [6]:
df = pd.read_csv('../data/raw/telco_churn.csv')

## 3. Custom Transformers

### 3.1 TotalCharges Cleaner

In [9]:
# =====================================================
# 3.1 TotalCharges Cleaner
# =====================================================

class TotalChargesCleaner(BaseEstimator, TransformerMixin):
    """
    Custom transformer to clean the TotalCharges column:
    - Convert to numeric (handling errors)
    - Apply domain logic: if tenure == 0 → TotalCharges = 0
    """
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Convert TotalCharges to numeric
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        
        # Apply domain logic: if tenure = 0 → TotalCharges = 0
        X.loc[X['tenure'] == 0, 'TotalCharges'] = 0
        
        
        return X

In [10]:
cleaner = TotalChargesCleaner()
cleaned_df = cleaner.transform(df)
print("\n✅ Test completed!")
print(f"Total missing values in TotalCharges after cleaning: {cleaned_df['TotalCharges'].isna().sum()}")


✅ Test completed!
Total missing values in TotalCharges after cleaning: 0


### 3.2 Feature Engineering

In [12]:
# =====================================================
# 3.2 Feature Engineering
# =====================================================

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Creates new features:
    - avg_monthly_spend
    - tenure_group
    - has_internet
    - has_addons
    - risky_contract
    - auto_payment
    """
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Average monthly spend
        X['avg_monthly_spend'] = np.where(
            X['tenure'] > 0,
            X['TotalCharges'] / X['tenure'],
            0
        )
        
        # Tenure group (customer lifecycle)
        bins = [0, 12, 36, float('inf')]
        labels = ['new', 'mid', 'loyal']
        X['tenure_group'] = pd.cut(X['tenure'], bins=bins, labels=labels, right=False)
        
        # Has internet
        X['has_internet'] = X['InternetService'].apply(lambda x: 0 if x == 'No' else 1)
        
        # Has addons (internet additional services)
        addon_cols = [
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies'
        ]
        X['has_addons'] = X[addon_cols].apply(
            lambda x: x.map({'Yes': 1, 'No': 0, 'No internet service': 0})
        ).sum(axis=1)
        
        # Risky contract (Month-to-month)
        X['risky_contract'] = X['Contract'].apply(
            lambda x: 1 if x == 'Month-to-month' else 0
        )
        
        # Auto payment
        auto_methods = ['Bank transfer (automatic)', 'Credit card (automatic)']
        X['auto_payment'] = X['PaymentMethod'].apply(
            lambda x: 1 if x in auto_methods else 0
        )
        
     
        return X

In [13]:
engineer = FeatureEngineer()
engineered_df = engineer.transform(cleaned_df)

print("\n" + "="*60)
print("✅ Feature Engineering Test Results:")
print(f"✓ New features created: {['avg_monthly_spend', 'tenure_group', 'has_internet', 'has_addons', 'risky_contract', 'auto_payment']}")
print(f"✓ avg_monthly_spend calculated correctly: {(engineered_df['avg_monthly_spend'] >= 0).all()}")
print(f"✓ tenure_group created: {engineered_df['tenure_group'].notna().all()}")
print(f"✓ has_addons range: {engineered_df['has_addons'].min()} - {engineered_df['has_addons'].max()}")
print(f"✓ risky_contract is binary: {engineered_df['risky_contract'].isin([0,1]).all()}")
print(f"✓ auto_payment is binary: {engineered_df['auto_payment'].isin([0,1]).all()}")
print("="*60)


✅ Feature Engineering Test Results:
✓ New features created: ['avg_monthly_spend', 'tenure_group', 'has_internet', 'has_addons', 'risky_contract', 'auto_payment']
✓ avg_monthly_spend calculated correctly: True
✓ tenure_group created: True
✓ has_addons range: 0 - 6
✓ risky_contract is binary: True
✓ auto_payment is binary: True


### 3.3 Сategorical encoder

In [15]:
# =====================================================
# 3.3 Categorical Encoder
# =====================================================

class CategoricalEncoder(BaseEstimator, TransformerMixin):
    """
    Encodes categorical features:
    - Binary features (Yes/No, Female/Male) → 0/1
    - Ordinal features: tenure_group and Contract
    - One-hot encoding for nominal features: MultipleLines, InternetService, PaymentMethod
    - Converts boolean columns to int
    """
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # 1. Binary encoding
        binary_cols = [
            'gender', 'Partner', 'Dependents', 
            'PhoneService', 'PaperlessBilling',
        ]
        
        for col in binary_cols:
            if col in X.columns:
                X[col] = X[col].map({
                    'Yes': 1, 
                    'No': 0, 
                    'Female': 1, 
                    'Male': 0
                })
        
        # 2. Ordinal encoding for tenure_group
        tenure_mapping = {'new': 0, 'mid': 1, 'loyal': 2}
        if 'tenure_group' in X.columns:
            X['tenure_group'] = X['tenure_group'].map(tenure_mapping)
            X['tenure_group'] = X['tenure_group'].astype(int)
        
        # 3. Ordinal encoding for Contract
        contract_map = {
            'Month-to-month': 0,
            'One year': 1,
            'Two year': 2
        }
        if 'Contract' in X.columns:
            X['Contract'] = X['Contract'].map(contract_map)
        
        # 4. One-hot encoding for multi-category features
        multi_cols = ['MultipleLines', 'InternetService', 'PaymentMethod']
        # Apply one-hot encoding
        X = pd.get_dummies(X, columns=multi_cols, drop_first=True)
        
        # 5. Convert boolean columns to int
        bool_cols = X.select_dtypes(include='bool').columns
        if len(bool_cols) > 0:
            X[bool_cols] = X[bool_cols].astype(int)
        
        # 6. Ensure only numeric columns remain
        X = X.select_dtypes(exclude='object')
        
        return X

In [16]:
encoder = CategoricalEncoder()
encoded_df = encoder.transform(engineered_df)

print("\nAfter Categorical Encoding:")
print("Columns:", encoded_df.columns.tolist())
print(f"Total columns after encoding: {encoded_df.shape[1]}")
print("Data types:\n", encoded_df.dtypes.value_counts())

print("\n" + "="*70)
print("✅ CategoricalEncoder Test Results:")
print("="*70)
print(f"✓ All columns are numeric: {encoded_df.select_dtypes(include='object').shape[1] == 0}")
print(f"✓ Binary columns encoded (0/1): {encoded_df['gender'].isin([0,1]).all() if 'gender' in encoded_df.columns else 'N/A'}")
print(f"✓ tenure_group is numeric: {pd.api.types.is_integer_dtype(encoded_df['tenure_group']) if 'tenure_group' in encoded_df.columns else 'N/A'}")
print(f"✓ One-hot encoding applied (example columns):")
onehot_example = [col for col in encoded_df.columns if col.startswith(('MultipleLines', 'InternetService', 'PaymentMethod'))]
print(f"   → {onehot_example[:6]} ...")  # показываем первые 6

print(f"\nFinal shape: {encoded_df.shape}")
print("All features are now numeric and ready for modeling.")


After Categorical Encoding:
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'Contract', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'tenure_group', 'has_internet', 'has_addons', 'risky_contract', 'auto_payment', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
Total columns after encoding: 23
Data types:
 int64      12
int32       8
float64     3
Name: count, dtype: int64

✅ CategoricalEncoder Test Results:
✓ All columns are numeric: True
✓ Binary columns encoded (0/1): True
✓ tenure_group is numeric: True
✓ One-hot encoding applied (example columns):
   → ['MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check'] ...

Final shape:

### 3.4 Drop Redundant Columns

In [18]:
# =====================================================
# 3.4 Drop Redundant Columns
# =====================================================

class DropRedundant(BaseEstimator, TransformerMixin):
    """
    Drops redundant and unnecessary columns:
    - customerID (identifier)
    - TotalCharges (correlated with tenure)
    - MonthlyCharges (duplicated by avg_monthly_spend)
    - num_services (duplicated by has_addons)
    - Contract (duplicated by risky_contract)
    """
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        cols_to_drop = [
            'customerID',        # unique identifier
            'TotalCharges',      # highly correlated with tenure
            'MonthlyCharges',    # duplicated by avg_monthly_spend
            'num_services',      # duplicated by has_addons
            'Contract'           # duplicated by risky_contract
        ]
        
        X = X.drop(columns=cols_to_drop, errors='ignore')
        
       
        
        return X

In [19]:
dropper = DropRedundant()
dropped_df = dropper.transform(encoded_df)

print("\nAfter dropping redundant columns:")
print(f"Number of columns: {dropped_df.shape[1]}")
print("Remaining columns:", dropped_df.columns.tolist())

print("\n✅ DropRedundant Test Results:")
print(f"✓ customerID removed: {'customerID' not in dropped_df.columns}")
print(f"✓ TotalCharges removed: {'TotalCharges' not in dropped_df.columns}")
print(f"✓ MonthlyCharges removed: {'MonthlyCharges' not in dropped_df.columns}")
print(f"✓ Final shape: {dropped_df.shape}")


After dropping redundant columns:
Number of columns: 20
Remaining columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'avg_monthly_spend', 'tenure_group', 'has_internet', 'has_addons', 'risky_contract', 'auto_payment', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

✅ DropRedundant Test Results:
✓ customerID removed: True
✓ TotalCharges removed: True
✓ MonthlyCharges removed: True
✓ Final shape: (7043, 20)


## 4. Scaling Numerical Features

In [21]:
# =====================================================
# 4. Scaling Numerical Features
# =====================================================

class NumericalScaler(BaseEstimator, TransformerMixin):
    """
    Scales numerical features using StandardScaler:
    - tenure
    - avg_monthly_spend
    - has_addons
    """
    def __init__(self):
        self.scaler = StandardScaler()
        self.scale_cols = ['tenure', 'avg_monthly_spend', 'has_addons']
    
    def fit(self, X, y=None):
        # Fit scaler only on the specified columns
        cols_to_scale = [col for col in self.scale_cols if col in X.columns]
        if cols_to_scale:
            self.scaler.fit(X[cols_to_scale])
        return self
    
    def transform(self, X):
        X = X.copy()
        cols_to_scale = [col for col in self.scale_cols if col in X.columns]
        
        if cols_to_scale:
            X[cols_to_scale] = self.scaler.transform(X[cols_to_scale])
   
        return X

In [22]:
scaler = NumericalScaler()
scaled_df = scaler.fit_transform(dropped_df)   


print("\nAfter scaling:")
print(scaled_df[['tenure', 'avg_monthly_spend', 'has_addons']].head(3))

print("\n✅ NumericalScaler Test Results:")
print(f"✓ Mean close to 0: {scaled_df['tenure'].mean():.4f}")
print(f"✓ Std close to 1: {scaled_df['tenure'].std():.4f}")
print(f"✓ All selected columns scaled successfully")


After scaling:
     tenure  avg_monthly_spend  has_addons
0 -1.277445          -1.151302   -0.561776
1  0.066327          -0.301458   -0.020519
2 -1.236724          -0.350966   -0.020519

✅ NumericalScaler Test Results:
✓ Mean close to 0: -0.0000
✓ Std close to 1: 1.0001
✓ All selected columns scaled successfully


## 5. Create Full Pipeline

In [24]:
# =====================================================
# 5. Create Full Pipeline
# =====================================================

# Define the final pipeline combining all transformers and the model
full_pipeline = Pipeline([
    # Preprocessing steps
    ('totalcharges_cleaner', TotalChargesCleaner()),
    ('feature_engineer', FeatureEngineer()),
    ('categorical_encoder', CategoricalEncoder()),
    ('drop_redundant', DropRedundant()),
    ('numerical_scaler', NumericalScaler()),
    
    # Final Model - Best CatBoost from tuning
    ('model', CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=4,
        l2_leaf_reg=50,
        colsample_bylevel=0.7,
        subsample=0.7,
        min_data_in_leaf=1,
        random_strength=5,
        class_weights=[1, 5],          # Best weight from tuning
        random_state=42,
        verbose=0
    ))
])

print("✅ Full sklearn Pipeline created successfully!")
print("Pipeline steps:")
for step in full_pipeline.named_steps.keys():
    print(f"   → {step}")

✅ Full sklearn Pipeline created successfully!
Pipeline steps:
   → totalcharges_cleaner
   → feature_engineer
   → categorical_encoder
   → drop_redundant
   → numerical_scaler
   → model


## 6. Train on Full Dataset

In [26]:
y = df['Churn'].map({'Yes': 1, 'No': 0})
full_pipeline.fit(df, y)

Pipeline(steps=[('totalcharges_cleaner', TotalChargesCleaner()),
                ('feature_engineer', FeatureEngineer()),
                ('categorical_encoder', CategoricalEncoder()),
                ('drop_redundant', DropRedundant()),
                ('numerical_scaler', NumericalScaler()),
                ('model',
                 CatBoostClassifier(class_weights=[1, 5], colsample_bylevel=0.7, depth=4, iterations=500, l2_leaf_reg=50, learning_rate=0.03, min_data_in_leaf=1, random_state=42, random_strength=5, subsample=0.7, verbose=0))])

## 7. Test Pipeline

In [28]:
get_metrics(y, full_pipeline.predict(df),full_pipeline.predict_proba(df)[:, 1])

{'Precision': 0.46377995642701525,
 'Recall': 0.9111824505082932,
 'F1-score': 0.6146904890813932,
 'Accuracy': 0.6968621326139429,
 'ROC-AUC': 0.8602616635054103}

### Model Performance Evaluation

The CatBoost model shows strong recall-oriented performance on the full dataset:

| Metric      | Value   | Interpretation                          |
|-------------|---------|-----------------------------------------|
| **Recall**    | 0.911   | Excellent – captures **91.1%** of churners |
| **ROC-AUC**   | 0.860   | Good discriminative power               |
| **F1-score**  | 0.615   | Moderate, due to recall-precision trade-off |
| **Precision** | 0.464   | Low – many false positives              |
| **Accuracy**  | 0.697   | Acceptable but less informative         |

**Summary**:  
This is a **high-recall model** designed to catch the majority of customers at risk of churn. It successfully identifies over 91% of churners, making it well-suited for retention campaigns, though at the expense of lower precision.

## 7. Save the Pipeline

In [ ]:
# Сохраняем полный пайплайн
model_path = 'models/full_churn_pipeline.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(full_pipeline, f)

print("✅ Pipeline successfully saved for production!")
print(f"📁 Saved at: {model_path}")
print(f"📊 File size: {os.path.getsize(model_path) / (1024*1024):.2f} MB")

## 8. How to Use the Saved Pipeline